# Colab 학습: AIHub 71363 변환 데이터 → 파인튜닝 → 제출 zip

**데이터 준비는 국내 PC에서 합니다** (AIHub 는 해외 IP 다운로드를 막아서 Colab 에서 직접 받을 수 없음).

국내 PC에서 할 일 (한 번만)
```
python scripts/merge_parts.py <받은폴더> --delete
python scripts/prepare_aihub71363.py --images <VS_02 원천> --labels <VL_01 라벨> --out data/aihub_val
python scripts/prepare_aihub71363.py --images <TS_02 원천> --labels <TL_01 라벨> --out data/aihub_train --windows 3 --max-pairs 1
```
`data/aihub_train`, `data/aihub_val` 을 각각 zip 으로 묶어 Google Drive 의 `knps/` 폴더에 올리고,
베이스라인 가중치 `unet_r18_cd.pt` 도 `knps/` 에 올립니다.

런타임 → 런타임 유형 변경 → **GPU(T4 이상)**.

In [ ]:
# [1] Drive 연결
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/knps'
import os; os.makedirs(f'{DRIVE}/runs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# [2] 코드 받기 + 패키지 (Colab 에는 CUDA torch 가 이미 있으므로 나머지만 설치)
!git clone -q -b claude/change-detection --depth 1 https://github.com/soomin0923/study /content/study || echo '비공개 레포면 Drive 에 코드 zip 을 올려 풀어 쓰십시오'
%cd /content/study/change_detection
!pip -q install segmentation-models-pytorch opencv-python-headless shapely nbformat nbconvert aifactory

In [ ]:
# [3] 데이터·가중치를 Colab 로컬 디스크로 (Drive 에서 직접 읽으면 학습이 매우 느림)
!mkdir -p data && unzip -q -o {DRIVE}/aihub_train.zip -d data/ && unzip -q -o {DRIVE}/aihub_val.zip -d data/
!cp {DRIVE}/unet_r18_cd.pt submission/assets/model/
!ls data; du -sh data/*
import glob; print('train', len(glob.glob('data/aihub_train/*/pre.png')), 'val', len(glob.glob('data/aihub_val/*/pre.png')))

**주의: 이 검증셋은 대회 점수의 대용품입니다.** 변화가 합성이라 '학습이 되고 있는지'는 알 수 있지만 대회 점수를 예측하지는 못합니다.
진짜 판단은 리더보드(먼저 디버그 제출)와 직접 라벨링한 검증셋으로 하십시오.

In [ ]:
# [4] 파인튜닝 - 체크포인트는 Drive 에 저장 (세션이 끊겨도 남음)
RUN = f'{DRIVE}/runs/aihub_r18_v1'
!python scripts/train.py --train data/aihub_train:1 --val data/aihub_val \
    --baseline submission/assets/model/unet_r18_cd.pt --cls-head \
    --epochs 20 --samples-per-epoch 6000 --bs 32 --workers 2 --out {RUN}

In [ ]:
# [5] 검증셋으로 판정 규칙 튜닝 (argmax 포함 클래스별 격자 탐색)
!python scripts/predict_folder.py --ckpt {RUN}/best.pt --pairs data/aihub_val --out {RUN}/val.npz --tta 1
!python scripts/tune_rules.py --probs {RUN}/val.npz --val data/aihub_val --out {RUN}/rules.json --top 5

In [ ]:
# [6] 제출 zip 만들기 -> Drive 에 복사
NAME = 'aihub-r18-v1'
!python scripts/build_submission.py --models {RUN}/best.pt --rules {RUN}/rules.json --tta 1 --name {NAME}
!cp dist/{NAME}.zip {DRIVE}/
print('Drive:', f'{DRIVE}/{NAME}.zip')

In [ ]:
# [7] 제출 (선택) - Colab Secrets 에 AIF_API_KEY 를 넣어두면 키 입력 없이 됩니다
DO_SUBMIT, DEBUG = False, True
if DO_SUBMIT:
    %load_ext aifactory
    os.chdir(f'dist/{NAME}')
    try:
        flag = '--debug' if DEBUG else ''
        %aifactory submit --notebook predict.ipynb --model-name {NAME} {flag}
    finally:
        os.chdir('/content/study/change_detection')